# Grouped stability: do the fusion findings survive unseen sequences?

The supplied mRNA stability split allows identical nucleotide sequences in training and test data. We keep all observations of each exact normalized sequence in one partition and refit the single-encoder and fusion heads. This tests whether their relative performance holds for unseen sequences, following up on the [encoder-swap findings on the supplied split](encoder_swaps.ipynb#Newton-three-seed-findings).

See how the models compare across seeds in the [grouped stability paired report](#Compare-the-three-seeds-without-mixing-uncertainty-sources).


## Setup

You can read the saved tables as shown. To refresh them, keep `READ_SAVED_RESULTS = True` and the training and comparison switches off. Use the kernel described in the [notebook environment setup](../../README.md#notebook-dependencies).

New fits use the stability CSV and the existing token caches. For where to find those inputs on Newton, see the [grouped stability execution guide](../../execution/arcc/grouped_stability/README.md).


In [1]:
# Paths, structured evidence and cleanup around opened cache files.
from dataclasses import asdict
from pathlib import Path
import sys

# Tables and array-level identity checks; import NumPy before PyTorch on Windows.
import numpy as np
import pandas as pd
import torch
from IPython.display import display

In [2]:
# Resolve the same Code/shared_code root from the repo or a notebook subdirectory.
START_DIR = Path.cwd().resolve()
CODE_DIR = next(
    (parent / "Code" for parent in (START_DIR, *START_DIR.parents)
     if (parent / "Code" / "shared_code").is_dir()),
    None,
)
if CODE_DIR is None:
    raise FileNotFoundError("Open this notebook from the project or its prepared workspace.")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code import datasets
from shared_code.fusion import blf, grouped_stability as grouped
from shared_code.fusion import run_identity, swap_reporting, token_embeddings as tokens, train

In [3]:
# Retain the existing BLF training recipe; changing the split is this experiment's intervention.
BASE_CONFIG = train.TrainConfig(
    lr=3e-5, weight_decay=1e-5, batch_size=32, max_epochs=500,
    early_stopping_patience=20, plateau_patience=5,
    channels=100, tconv="equation", entropy_lambda=0.0, seed=0,
)
# MIL alone uses the inherited entropy penalty 0.01; no new penalty search.
PANEL = grouped.fit_panel()

In [ ]:
# Seed roles differ: one partition, three neural repeats, one baseline-selection rule.
SPLIT_SEED = 42
TRAINING_SEEDS = (0, 1, 2)
SELECTION_SEED = 0
SEED = 0  # The one training repeat assigned to a Newton job.
BOOTSTRAP_SEED = 0
BOOTSTRAP_DRAWS = 2000

READ_SAVED_RESULTS = True
RUN_TRAINING = False
RUN_COMPARISONS = False  # Rescores saved predictions and writes a separate paired report.

DATA_DIR = CODE_DIR.parent / "datasets"
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "grouped_stability_v1"
DEVICE = "cpu"  # Reading uses no GPU; the Newton launcher verifies and injects CUDA.

# The launcher supplies observed execution provenance and cohort-aware reuse checks.
FIT_PRODUCER = None
RESULT_VALIDATOR = None
CHECKPOINT_VALIDATOR = None

In [5]:
if RUN_TRAINING and (READ_SAVED_RESULTS or RUN_COMPARISONS):
    raise ValueError("Fitting and saved-prediction analysis are separate operations.")
if not READ_SAVED_RESULTS and not RUN_TRAINING:
    raise ValueError("Choose saved reading or the explicit Newton training route.")
if (SPLIT_SEED, TRAINING_SEEDS, SELECTION_SEED) != (42, (0, 1, 2), 0):
    raise ValueError("This approved protocol uses split 42, training 0/1/2 and selection 0.")
if RUN_TRAINING:
    if not all(callable(hook) for hook in (FIT_PRODUCER, RESULT_VALIDATOR, CHECKPOINT_VALIDATOR)):
        raise ValueError("Use the prepared Newton launcher to supply execution and reuse checks.")
    if DEVICE != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("Newton fitting requires CUDA; no automatic CPU fallback.")
    print(f"Training seed {SEED}: {torch.cuda.get_device_name(0)}")
else:
    print("Saved-results mode: no partitions, cache arrays or prediction heads are computed.")

Saved-results mode: no partitions, cache arrays or prediction heads are computed.


## Which comparisons answer the question?

Keeping identical sequences together before refitting follows the stability reliability check in [Mina's plan, section A.1](../../docs/Biological_Fusion_Plan.docx).

The original combination uses NT-v2 100M, RNA-FM and ESM-2 8M. The second replaces RNA-FM with mRNA-FM while retaining DNA and protein encoders. For each combination, compare concatenation, MIL gating and cross-attention against eligible single-input controls and against each other.

The six unique controls are native NT-v2, RNA-FM, mRNA-FM and ESM-2, plus NT-v2 and RNA-FM after codon-grid preparation. RNA-FM pooling and NT-v2's learned upsampling change the information supplied to the prediction head; their controls prevent automatically attributing a benefit to combination alone. mRNA-FM codons and ESM-2 residues already use the relevant resolution.

This gives twelve fits per training seed, thirty-six in total. Shared single-input controls are not fitted again for the second combination. There are no out-of-fold correction models in this experiment.

In [6]:
display(pd.DataFrame([
    {"fit": spec.key, "predictor": spec.variant, "encoder combination": spec.bundle,
     "streams read by the predictor": ", ".join(spec.modalities)}
    for spec in PANEL
]))
print(f"Declared workload: {len(PANEL)} fits × {len(TRAINING_SEEDS)} seeds")

,fit,predictor,encoder combination,streams read by the predictor
0,ntv2_native,unimodal:dna,original,dna
1,ntv2_grid,unimodal-grid:dna,original,dna
2,rnafm_native,unimodal:rna,original,rna
3,rnafm_grid,unimodal-grid:rna,original,rna
4,mrnafm_native,unimodal:rna,mrnafm,rna
5,esm2_native,unimodal:prot,original,prot
6,original_concat,concat,original,"dna, rna, prot"
7,original_mil,mil,original,"dna, rna, prot"
8,original_cross,cross,original,"dna, rna, prot"
9,mrnafm_concat,concat,mrnafm,"dna, rna, prot"


Declared workload: 12 fits × 3 seeds


## Keep observations together without changing their targets

Repeated observations of the same sequence can have different stability scores, so grouping keeps their original targets. The dataset guide explains how the scores are [standardized within experimental profiles](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles).

Retain the existing ≤999-nucleotide cohort and its original CSV order. For group identity only, trim surrounding whitespace, uppercase letters and treat T/U as equivalent. Hash each canonical nucleotide sequence, then use a seeded hash bucket to assign whole groups to approximately 70% training, 15% validation and 15% test. Group sizes make the exact row proportions approximate. Do not search split seeds for favorable results or stratify on target values in this first comparison.

`SPLIT_SEED = 42` follows existing Stage 1 splitting conventions; it has no scientific privilege. Neural seeds 0/1/2 repeat initialization and training on the same partition. Selection seed 0 determines one single-input baseline on the new validation partition, retained across all three training repeats. Bootstrap seed 0 controls later test-group resampling. Historical notebooks keep their own recorded seed choices.

The saved split record identifies source rows, group assignments, targets and source bytes. The original `Split` column and CSV remain unchanged; training uses the separate `GroupedSplit` column. Existing records are reused only if they match exactly. Each Newton seed job independently derives the same assignment and can publish it once without overwriting another job's record.

In [7]:
SPLIT_PATH = RESULTS_ROOT / "split.json"
benchmark_rows = None
split_record = None
if RUN_TRAINING:
    grouped.require_result_destination(RESULTS_ROOT, CACHE_ROOT)
    source_path = DATA_DIR / "mRNA_Stability.csv"
    source_rows = datasets.blf_benchmark_rows("mrna_stability", data_dir=DATA_DIR)
    benchmark_rows, split_record = grouped.build_split(
        source_rows, source_sha256=tokens.file_fingerprint(source_path), split_seed=SPLIT_SEED,
    )
    written = grouped.publish_split(SPLIT_PATH, split_record)
    print("Grouped assignment saved." if written else "Matching grouped assignment reused.")
elif SPLIT_PATH.is_file():
    split_record = grouped.read_split(SPLIT_PATH, split_seed=SPLIT_SEED)
else:
    print("No saved grouped assignment yet. Reading mode does not create one.")

In [8]:
if split_record is not None:
    display(pd.DataFrame.from_dict(split_record["counts"], orient="index"))
    print("Source SHA-256:", split_record["source_sha256"])
    print("Grouped assignment:", split_record["fingerprint"])

,rows,groups
train,16940,7954
val,3493,1662
test,3496,1687


Source SHA-256: 50d92188f7e9d2111e526f6d520ddb28d20bd7bb19190d23e011c18c8d660dbb
Grouped assignment: e42adf9630bc0e6b99e900036d0004030150593cdef9fc767411cceaa9682ea7


## Read completed fits before scheduling new work

Reading checks the declared split, complete training configuration, encoder identity and saved prediction/target correspondence. Missing results are reported; incompatible results block fitting rather than being replaced. The saved scores describe observed fits, and a missing record is not evidence that a model performed poorly.

The default reader covers all three seeds. A Newton training job checks and writes only its assigned seed, under `fits/<fit>/seedN.json`. The new root has no fallback to the historical desktop or supplied-split Newton results.

In [9]:
requested_seeds = (SEED,) if RUN_TRAINING else TRAINING_SEEDS
records, issues = {}, []
if split_record is not None:
    records, issues = grouped.read_results(
        RESULTS_ROOT, split_record, BASE_CONFIG, seeds=requested_seeds,
        validator=RESULT_VALIDATOR,
    )
    if issues:
        display(pd.DataFrame(issues))
    display(pd.DataFrame([
        {"fit": key, "seed": seed, "validation Spearman": entry["data"]["val_score"],
         "test Spearman": entry["data"]["test_score"]}
        for (key, seed), entry in records.items()
    ]))
    print(f"Compatible saved fits: {len(records)}/{len(PANEL) * len(requested_seeds)}")

,fit,seed,validation Spearman,test Spearman
0,ntv2_native,0,0.256683,0.259813
1,ntv2_grid,0,0.285125,0.290609
2,rnafm_native,0,0.316769,0.309939
3,rnafm_grid,0,0.301731,0.290922
4,mrnafm_native,0,0.395266,0.371600
5,esm2_native,0,0.317194,0.301705
6,original_concat,0,0.358449,0.326029
7,original_mil,0,0.347070,0.319139
8,original_cross,0,0.346201,0.313033
9,mrnafm_concat,0,0.401260,0.375940


Compatible saved fits: 36/36


## Fit only the missing predictors using existing caches

The four reused caches are `ntv2_100m`, `rnafm`, `mrnafm` and `esm2_8m`. Metadata and shard hashes establish input compatibility before any model is built. Missing or incompatible caches stop the job; this route has no extraction or model-download call. The token loader reads only the modalities required by each predictor.

All heads start fresh for this grouped split. A checkpoint may resume only within the same identified experiment and configuration. Completed fits retain their recorded predictions and weights. The following definitions make preparation and execution visible separately; defining them does not run training.

In [10]:
def verified_cache_records(rows):
    """Read and verify complete cache manifests without modifying their contents."""
    verification = tokens.CacheVerification()
    encoder_keys = sorted({encoder for spec in PANEL for encoder in spec.encoders.values()})
    metadata = {}
    for encoder_key in encoder_keys:
        metadata[encoder_key] = run_identity.cache_record(
            CACHE_ROOT / "mrna_stability" / encoder_key, verification=verification,
        )
    return metadata

In [11]:
def open_fit_streams(rows, spec):
    """Read cache lengths now; token arrays are accessed only by the selected streams."""
    caches = {}
    try:
        for modality, encoder_key in spec.encoders.items():
            caches[modality] = tokens.summarize(CACHE_ROOT / "mrna_stability" / encoder_key)
        streams = blf.BenchmarkStreams(
            "mrna_stability", rows, caches, "regression", "Value", grouped.SPLIT_COLUMN,
            encoders=spec.encoders,
        )
        streams.check()
        return streams
    except BaseException as error:
        for cache in caches.values():
            try:
                cache.close()
            except BaseException as cleanup_error:
                error.add_note(f"Cache cleanup also failed: {cleanup_error}")
        raise

In [12]:
def fit_one(rows, spec, config, identity):
    """Fit one missing/resumable head and release streams even if the fit fails."""
    path = grouped.fit_path(RESULTS_ROOT, spec, config.seed)
    streams = open_fit_streams(rows, spec)
    primary_error = None
    try:
        print(f"Grouped stability / {spec.key} / seed {config.seed}", flush=True)
        return train.run(
            streams, spec.variant, config, DEVICE, path.parent, tag=path.stem,
            run_identity=identity, producer=FIT_PRODUCER(), require_predictions=True,
        )
    except BaseException as error:
        primary_error = error
        raise
    finally:
        try:
            streams.close()
        except BaseException as cleanup_error:
            if primary_error is None:
                raise
            primary_error.add_note(f"Stream cleanup also failed: {cleanup_error}")

In [13]:
def preflight_missing_fits(rows, split, completed, problems):
    """Check every requested identity and checkpoint before starting any new fit."""
    invalid = [issue for issue in problems if issue["issue"] != "missing"]
    if invalid:
        raise ValueError(f"Incompatible results are preserved: {invalid}")
    pending = [spec for spec in PANEL if (spec.key, SEED) not in completed]
    if not pending:
        return []
    metadata = verified_cache_records(rows)
    planned = []
    for spec in PANEL:
        config = grouped.config_for(spec, BASE_CONFIG, SEED)
        identity = grouped.make_identity(rows, split, spec, config, metadata)
        if (spec.key, SEED) in completed:
            if completed[spec.key, SEED]["data"]["identity"] != identity:
                raise ValueError(f"{spec.key}: completed cache/model identity differs; preserve it.")
            continue
        path = grouped.fit_path(RESULTS_ROOT, spec, SEED)
        if path.with_suffix(".weights.pt").exists():
            raise FileExistsError(f"Orphan weights need inspection; preserve {path}.")
        checkpoint = path.with_suffix(".checkpoint")
        if checkpoint.exists():
            CHECKPOINT_VALIDATOR(checkpoint, identity)
        planned.append((spec, config, identity))
    return planned

In [14]:
if RUN_TRAINING:
    planned = preflight_missing_fits(benchmark_rows, split_record, records, issues)
    print(f"Seed {SEED}: {len(planned)} missing/resumable fits; completed fits are preserved.")
    for spec, config, identity in planned:
        fit_one(benchmark_rows, spec, config, identity)
    records, issues = grouped.read_results(
        RESULTS_ROOT, split_record, BASE_CONFIG, seeds=(SEED,), validator=RESULT_VALIDATOR,
    )
    if issues or len(records) != len(PANEL):
        raise RuntimeError(f"Seed {SEED} remains incomplete: {issues}")
    print(f"Requested grouped seed {SEED} completed: {len(records)}/{len(PANEL)} fits.")
else:
    print("Training disabled.")

Training disabled.


## Select baselines using the new validation partition

Select the highest seed-0 validation Spearman within each encoder combination and across the full declared single-input panel. Eligible single-input controls include native and grid-prepared inputs. Fix each selected configuration across seeds 0/1/2; do not inherit the supplied-split winner or select on test scores. Ties follow the declared panel order. An incomplete or undefined validation comparison does not silently fall back to the available subset.

The across-panel comparison prevents a gain against a weaker within-combination baseline from implying superiority to the strongest declared single. MIL and cross-attention must also face their same-input concatenation baseline. No new hyperparameter sweep or automatic expansion to other encoders is included.

In [15]:
selected_singles = None
if split_record is not None and not RUN_TRAINING:
    try:
        selected_singles, comparisons = grouped.selection_and_comparisons(
            records, selection_seed=SELECTION_SEED,
        )
    except ValueError as error:
        print(f"Selection unavailable: {error}")
    else:
        display(pd.DataFrame([
            {"comparison scope": scope, "selected single": key}
            for scope, key in selected_singles.items()
        ]))

,comparison scope,selected single
0,original,esm2_native
1,mrnafm,mrnafm_native
2,all,mrnafm_native


## Compare the three seeds without mixing uncertainty sources

`RUN_COMPARISONS = True` enables a separate saved-prediction analysis after all thirty-six compatible fits are present. It averages the three fitted scores rather than ensembling predictions. The existing paired-report implementation resamples whole test-sequence groups and reports a 95% interval for each paired difference, conditional on the fitted seeds. The standard deviation across the three seed differences describes a different uncertainty source. Intervals remain exploratory and unadjusted for multiple comparisons.

Keep this switch false when reading an existing report or training a Newton seed. The report binds its input fits and grouped partition; reading verifies those bindings without recalculating scores. The default output root has no recorded results until an authorized run returns them.

In [16]:
saved_report, report_path = None, None
if split_record is not None and not RUN_TRAINING:
    if RUN_COMPARISONS:
        bundle = grouped.report_bundle(records, issues, BASE_CONFIG)
        test_positions = np.flatnonzero(np.asarray(split_record["partitions"]) == "test")
        groups = np.asarray(split_record["groups"])[test_positions]
        saved_report, report_path, written = swap_reporting.report(
            bundle, truth=np.asarray(split_record["test_truth"], dtype=np.float32),
            groups=groups, task="regression", context=grouped.report_context(split_record),
            output_dir=RESULTS_ROOT / "reports", draws=BOOTSTRAP_DRAWS,
            bootstrap_seed=BOOTSTRAP_SEED, repository=CODE_DIR.parent,
        )
        print("Paired report calculated." if written else "Identical paired report reused.")
    elif not issues:
        saved_report, report_path = grouped.read_report(
            RESULTS_ROOT, records, split_record, BASE_CONFIG,
            draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED,
        )
if saved_report is None:
    print("No eligible saved report displayed; no replacement analysis was run.")
else:
    print(report_path)
    display(pd.DataFrame(saved_report["analysis"]["seed_summary"]))

Paired report calculated.
D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\grouped_stability_v1\reports\mrna_stability__0c3daa636261f77c1f10.json


,encoder,kind,comparison,first,second,available_seeds,missing_seeds,status,first_score,second_score,difference,seed_sd,ci_low,ci_high,first_wins,valid_draws,requested_draws,interval_issue
0,original,complementarity,concat vs selected single (original),original_concat,esm2_native,"[0, 1, 2]",[],complete,0.327705,0.304524,0.023181,0.008241,-0.008591,0.053335,0.9245,2000,2000,
1,original,complementarity,concat vs selected single (all),original_concat,mrnafm_native,"[0, 1, 2]",[],complete,0.327705,0.377124,-0.049419,0.016604,-0.080847,-0.018137,0.0010,2000,2000,
2,original,complementarity,mil vs selected single (original),original_mil,esm2_native,"[0, 1, 2]",[],complete,0.312347,0.304524,0.007822,0.013665,-0.012872,0.026604,0.7765,2000,2000,
3,original,complementarity,mil vs selected single (all),original_mil,mrnafm_native,"[0, 1, 2]",[],complete,0.312347,0.377124,-0.064777,0.011015,-0.096411,-0.031270,0.0000,2000,2000,
4,original,complementarity,cross vs selected single (original),original_cross,esm2_native,"[0, 1, 2]",[],complete,0.321848,0.304524,0.017323,0.005646,-0.013649,0.045811,0.8615,2000,2000,
5,original,complementarity,cross vs selected single (all),original_cross,mrnafm_native,"[0, 1, 2]",[],complete,0.321848,0.377124,-0.055276,0.006524,-0.086268,-0.023578,0.0005,2000,2000,
6,original,method,mil vs concat,original_mil,original_concat,"[0, 1, 2]",[],complete,0.312347,0.327705,-0.015359,0.020384,-0.036571,0.007315,0.0910,2000,2000,
7,original,method,cross vs concat,original_cross,original_concat,"[0, 1, 2]",[],complete,0.321848,0.327705,-0.005857,0.012097,-0.024514,0.013001,0.2605,2000,2000,
8,mrnafm,complementarity,concat vs selected single (mrnafm),mrnafm_concat,mrnafm_native,"[0, 1, 2]",[],complete,0.376786,0.377124,-0.000338,0.007401,-0.011115,0.011518,0.4890,2000,2000,
9,mrnafm,complementarity,concat vs selected single (all),mrnafm_concat,mrnafm_native,"[0, 1, 2]",[],complete,0.376786,0.377124,-0.000338,0.007401,-0.011115,0.011518,0.4890,2000,2000,


## What would these results establish?

Compare fusion with the new validation-selected singles, then compare attention and gating with the matching concatenation. If an advantage persists across seeds under this split, it supports the corresponding prediction claim for unseen exact nucleotide sequences in this retained benchmark. If it weakens or reverses, revise that claim before using it to motivate additional fusion mechanisms. An inconclusive interval is not proof of equivalence.

The old and new test populations differ, so subtracting their scores does not isolate a causal amount of leakage inflation. This is a retrospective sensitivity experiment: previous benchmark findings informed the panel. It is not an independent prospective validation, and exact grouping does not resolve pretraining exposure or related-sequence overlap.

Near-identical sequence clusters, annotated gene families, species and experimental-condition holdouts are possible future questions, not claims established here. The stability CSV omits source-profile identifiers; a reliable recovery from original records would be needed for condition/species allocation. Sequence-similarity clustering can test transfer beyond close relatives, but is not automatically an annotated gene-family split. Synonymous nucleotide strings can still encode the same protein.

The outcome informs whether to trust stability as evidence while developing [Mina's auxiliary-correction investigation](README.md#can-auxiliary-features-correct-the-anchors-errors). That later experiment has its own protocol and out-of-fold fits. No IsoFormer, LucaOne or Coupled Mamba reproduction is required for this bounded comparison.